In [1]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
current_dir = '/content/drive/MyDrive/Colab Notebooks/Tesi'
%cd $current_dir

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/Tesi


In [2]:
!pip install -q imodels

In [4]:
import random
import numpy as np
import pandas as pd
from imodels import SkopeRulesClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

df = pd.read_csv("dataset_clusters_for_skope_B_all_R1.csv")
df["Cluster"] = df["Cluster"].astype(str)

y_os = df["OS2ys"].values
clusters = df["Cluster"]
X = df.drop(columns=["Cluster", "OS2ys"])

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    X[cat_cols] = X[cat_cols].astype(str)
    X = pd.get_dummies(X, columns=cat_cols, dtype=int)

X = X.reindex(sorted(X.columns), axis=1)
feature_names = list(X.columns)

PREC_MIN, REC_MIN = 0.45, 0.20
res_all = []

for cl_id in sorted(clusters.unique(), key=lambda s: int(s)):
    y_bin = (clusters == cl_id).astype(int).values
    n_pos = y_bin.sum()

    sk = SkopeRulesClassifier(
        n_estimators=50,
        max_depth=(2, 3),
        precision_min=PREC_MIN,
        recall_min=REC_MIN,
        bootstrap=True,
        max_samples=1.0,
        min_samples_split=10,
        random_state=SEED,
    )
    sk.fit(X.values, y_bin, feature_names=feature_names)

    rows = []
    for r in sk.rules_:
        rule_str = getattr(r, "rule", str(r))
        try:
            match = X.eval(rule_str).values.astype(bool)
        except Exception:
            continue
        if match.sum() == 0:
            continue
        tp = (match & (y_bin == 1)).sum()
        prec = tp / match.sum()
        rec = tp / n_pos
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

        rows.append({
            "Cluster": cl_id,
            "Identified_Rule": rule_str,
            "Precision": round(prec, 3),
            "Recall": round(rec, 3),
            "F1_Score": round(f1, 3),
            "N_Matched": int(match.sum()),
            "OS2ys_in_Rule": round(y_os[match].mean(), 3),
        })

    if rows:
        tab = (pd.DataFrame(rows)
               .drop_duplicates("Identified_Rule")
               .query("Precision >= @PREC_MIN and Recall >= @REC_MIN")
               .sort_values(["F1_Score", "Precision"], ascending=False)
               .head(2))
        res_all.append(tab)

if res_all:
    #tabella_regole = pd.concat(res_all, ignore_index=True)
    #print(tabella_regole.to_string(index=False))

    tabella_top1 = tabella_regole.groupby("Cluster", sort=False).head(1)
    print(tabella_top1.to_string(index=False))
else:
    print("no rule found with those precision and recall filters")

Cluster                                           Identified_Rule  Precision  Recall  F1_Score  N_Matched  OS2ys_in_Rule
      1   Age > 71.5 and LAA <= 0.908057 and Volume_PTV <= 288.16      1.000   0.964     0.981         53          0.623
      2                   Age <= 72.5 and Volume_PTV <= 289.36501      0.963   0.890     0.925        109          0.725
      3 Age <= 75.5 and LAA <= 1.51067 and Volume_PTV > 289.85501      1.000   1.000     1.000        169          0.550
      4                     LAA > 1.51067 and Volume_PTV > 289.59      1.000   1.000     1.000         28          0.179
      5     Age > 75.5 and LAA <= 1.57607 and Volume_PTV > 299.42      1.000   1.000     1.000         37          0.270


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
